In [1]:
import sys
sys.path.append("/Users/sujaladhikari/Sujal's Personal/Projects/FedIDS")

In [2]:
import os 
import shutil
import numpy as np 
import pandas as pd 
from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset
import torch 
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from Model.model import MLP
from torch.optim import Adam
import utils
from utils import JoinCustomDataset
from sklearn.metrics import classification_report
from federatedlearning import updatefrom_local, weight_averaging

### Setting up the device

In [3]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
device

device(type='mps')

### Creating the global model - using the same MLP used for the centralized model 

In [4]:
input_size = 78
hidden_layer = [256, 128,64,8]
num_classes = 2
global_model = MLP(input_size, hidden_layer,num_classes).to(device)
global_model
num_clients = 4

### Creating two folders that holds the training input data, and the other holds the result 

In [5]:
manual = ['One', 'Two', 'Three' , 'Four']
types = ['X_train', 'y_train', 'X_val', 'y_val', 'X_test', 'y_test']
source_directory = '../silos_datasets/'
destination_directory  = './client_data/nids/' ## We are creating sub-directory in order to check on the future datasets.
os.makedirs(destination_directory , exist_ok=True)
for index,a in enumerate(manual):
    new_client_name = f'client_{index}'
    for type in types:
        old_file = f'silo{a}_{type}.csv'
        old_filepath = os.path.join(source_directory, old_file)
        
        new_file = f'{new_client_name}_{type}.csv'
        new_filepath = os.path.join(destination_directory , new_file)

        if os.path.exists(old_filepath):
            shutil.copy(old_filepath, new_filepath)

saving_directory = os.path.join('./output_nids/DNN_fedavg/nids') ## Creating the folder that stores the result of each client performance 
os.makedirs(saving_directory, exist_ok=True)

### Creating the Data Configuration and Training Configuration 


In [6]:
batch_size = 64 ## Initially we set up as same as the centralized model 
lr = 0.001 ## The learning rate is same as that of the centralized model 
num_rounds = 5 ## 5/.001 => 5000 rounds 
num_local_epochs = 5
save_interval = 1

### We will be creating new dataset which contains all the testing data, that will keep on checking on how the global model is performing !?

### This will be the golden dataset that will contain 50% of the randomly sampled data from each of the testing data of the clients, and to make it more further we will be shuffling the data and will take the 100% of the whole shuffle data !

In [7]:
## Creating an ultimate gloden dataset 
golden_data = []
path = './client_data/nids/'
for i in range(num_clients):
    filepath = os.path.join(path, f'client_{i}_X_test.csv')
    testfilepath = os.path.join(path, f'client_{i}_y_test.csv')
    if os.path.exists(filepath) and os.path.exists(testfilepath):
        x_test_data = pd.read_csv(filepath)
        test_data = pd.read_csv(testfilepath)
        combined_data = pd.concat([x_test_data,test_data], axis = 1)
        golden_data.append(combined_data)


global_dataset = pd.concat(golden_data, ignore_index=True)
## Randomly shuffling the dataset 
global_dataset = global_dataset.sample(frac = 1, random_state = 42).reset_index(drop = True)
global_dataset.head(5)

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label_Binary
0,5002,43,1,1,2,6,2,2,2.000000,0.000000,...,24,0.00000,0.00000,0,0,0.0,0.00000,0,0,1
1,443,60414768,14,16,718,4337,367,0,51.285714,110.881384,...,32,40929.66667,39340.51837,121233,24725,10000000.0,11414.67646,10000000,9994667,0
2,53,255,2,2,60,344,30,30,30.000000,0.000000,...,32,0.00000,0.00000,0,0,0.0,0.00000,0,0,0
3,545,42,1,1,2,6,2,2,2.000000,0.000000,...,24,0.00000,0.00000,0,0,0.0,0.00000,0,0,1
4,80,113616,3,0,0,0,0,0,0.000000,0.000000,...,32,0.00000,0.00000,0,0,0.0,0.00000,0,0,0


In [8]:
X = global_dataset.drop(columns = ['Label_Binary']).values
y = global_dataset['Label_Binary'].values

X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype = torch.long)


global_tuple = (X_tensor,y_tensor)

### Two Global Metrics for the global model 

In [9]:
performance_dict, performance_log = dict(), dict()
metric_keys = ['g_train_loss', 'g_test_loss']
performance_dict, performance_log = utils.performance_analyzer(metric_keys)

### Automatically resuming from the checkpoint 


In [10]:
# Checking if there is already anything going on 
log_path = os.path.join(saving_directory, 'performacnce_log.pickle')
if os.path.isfile(log_path):
    performance_log = utils.loading_pickle(log_path)
starting_round = len(performance_log[metric_keys[0]])
if starting_round > 0:
    global_model.lo

In [11]:
labmda = 0.01

### Loading all the client data before training them to the global model 


In [12]:
client_loaders = []
for index in range(num_clients):
    features_path = f'client_{index}_X_train.csv'
    labels_path = f'client_{index}_y_train.csv'
    features_directory = os.path.join(destination_directory, features_path )
    labels_directory = os.path.join(destination_directory, labels_path)
    print(features_directory,labels_directory)
    dataset = utils.JoinCustomDataset(features_directory, labels_directory)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size = batch_size, shuffle = True)
    client_loaders.append(dataloader)


./client_data/nids/client_0_X_train.csv ./client_data/nids/client_0_y_train.csv
./client_data/nids/client_1_X_train.csv ./client_data/nids/client_1_y_train.csv
./client_data/nids/client_2_X_train.csv ./client_data/nids/client_2_y_train.csv
./client_data/nids/client_3_X_train.csv ./client_data/nids/client_3_y_train.csv


### Creating a test loader for each client to check the test loss 

In [13]:
validation_loaders = []
for index in range(num_clients):
    features_path = f'client_{index}_X_val.csv'
    labels_path = f'client_{index}_y_val.csv'
    features_directory = os.path.join(destination_directory, features_path )
    labels_directory = os.path.join(destination_directory, labels_path)
    print(features_directory,labels_directory)
    dataset = utils.JoinCustomDataset(features_directory, labels_directory)
    dataloader = torch.utils.data.DataLoader(dataset, batch_size = batch_size, shuffle = True)
    validation_loaders.append(dataloader)

./client_data/nids/client_0_X_val.csv ./client_data/nids/client_0_y_val.csv
./client_data/nids/client_1_X_val.csv ./client_data/nids/client_1_y_val.csv
./client_data/nids/client_2_X_val.csv ./client_data/nids/client_2_y_val.csv
./client_data/nids/client_3_X_val.csv ./client_data/nids/client_3_y_val.csv


In [15]:
global_weights = global_model.state_dict() ## This gives the initial weights of the given model
loss_function = nn.CrossEntropyLoss()
optimization_args = {'lr':lr}

### Starting of the model 

### We wll be looping the model from start round to number of rounds
### Here the starting round is initially zero as nothing is loaded in the given model 

for round_number in range(1): ## Initially 0 -> 5
    global_model.train()
    client_updates = dict() ## We will be storing the updates given by clients in a dictionary 
    for client_number in range(1):
        print('Client:', client_number)
        client_loader = client_loaders[client_number]
        validation_loader = validation_loaders[client_number]
        client_update = updatefrom_local(global_model,client_loader, validation_loader, num_local_epochs, optimization_args)
        client_updates.setdefault('local_weights', list()).append(client_update['local_weights'])
        client_updates.setdefault('num_samples', list()).append(client_update['num_samples'])

        ## Performance log update 
        performance_log.setdefault('c_{}_train_loss'.format(client_number), list()).append(client_update['train_loss'])
        performance_log.setdefault('c_{}_test_loss'.format(client_number), list()).append(client_update['test_loss'])

    
    global_weights = weight_averaging(client_updates['local_weights'], client_updates['num_samples'], device)
    print(global_weights)

Client: 0


Local Testing Loss: 100%|██████████| 1180/1180 [00:01<00:00, 998.56it/s] 


OrderedDict([('fc_layers.0.weight', tensor([[ 1.3947,  0.0815, -0.6639,  ..., -0.1061, -0.0420,  0.0548],
        [ 1.3036, -0.1301, -2.0308,  ..., -0.0147, -0.0225, -0.1593],
        [ 0.8084,  0.2522,  0.3309,  ..., -0.0550,  0.0870,  0.1233],
        ...,
        [ 1.3011,  0.0334,  0.3515,  ..., -0.2632,  0.1042,  0.0124],
        [ 0.3217,  0.0603, -0.1138,  ...,  0.0554,  0.0950,  0.1319],
        [ 0.0523,  0.0399,  0.8978,  ..., -0.1716, -0.1955, -0.0235]],
       device='mps:0')), ('fc_layers.0.bias', tensor([-2.9673e-01, -1.4679e+00,  4.1278e-01,  2.0407e+00, -2.9571e+00,
        -1.3245e-03, -5.3047e-01,  7.0445e-01,  3.3530e+00,  7.2545e-01,
        -5.8694e-01, -3.8244e-02, -2.8924e+00, -3.8234e-02, -1.0375e-01,
         6.2128e-01,  1.4871e+00, -3.3901e-01, -8.8856e-02,  4.2623e+00,
         5.2597e-01,  1.5884e-01, -1.0244e+00,  9.8217e-01,  2.7250e+00,
         4.5760e-01,  1.7940e+00,  7.8151e-01,  1.9994e-01, -3.6877e-01,
        -3.0600e+00,  4.1665e+00,  3.2039e-01,

### Why is it using 5507 samples of data, and how to know what is going on in the output!